<a href="https://colab.research.google.com/github/AnastasiyaPunko/Cygnus-Test-Pipeline/blob/main/NIPT_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
conda activate ngs

In [ ]:
conda list -n ngs

QC

In [ ]:
fastqc nipt_raw.fq

In [ ]:
trimmomatic SE -phred33 nipt_raw.fq nipt_raw_trimm.fq.gz ILLUMINACLIP:/home/punko_a/genom/adapter.fa:2:30:10 LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

In [ ]:
cutadapt -aadapter_seq -e 0.15 -m 35 -q 20 -o nipt_trimmed.fastq.gz nipt_raw_trimm.fq.gz

In [ ]:
fastqc nipt_trimmed.fastq.gz

In [ ]:
bwa mem -R '@RG\tID:GSS5-0798\tSM:Hemato_pro_540_chip\tPL:ILLUMINA' -t 3 /home/punko_a/genom/Homo_sapiens_assembly37.fasta nipt_trimmed.fastq.gz -o nipt.sam

In [ ]:
samtools view -Sb nipt.sam > nipt.bam

In [ ]:
samtools sort -m 3G -o nipt_sort.bam nipt.bam

In [ ]:
samtools index nipt_sort.bam

On-target quantification

In [ ]:
bedtools coverage -a targets.bed -b nipt_sort.bam > target_counts.tsv

In [ ]:
awk '{print $4"\t"$NF}' target_counts.tsv > counts.tsv

Calculate GC content for each probe

In [ ]:
bedtools nuc -fi /home/punko_a/genom/Homo_sapiens_assembly37.fasta -bed targets.bed > gc_content.tsv

Remove systematic GC bias

In [ ]:
import pandas as pd
from statsmodels.nonparametric.smoothers_lowess import lowess

counts = pd.read_csv("counts.tsv", sep="\t", names=["probe","count"])
gc = pd.read_csv("gc_content.tsv", sep="\t")

df = counts.merge(gc, on="probe")

fit = lowess(df["count"], df["GC"], frac=0.3)

df["expected"] = fit[:,1]
df["corrected"] = df["count"] / df["expected"]

df.to_csv("corrected_counts.tsv", sep="\t", index=False)

Create reference file

In [ ]:
import pandas as pd
import glob

files = glob.glob("reference_build/*.tsv")

dfs = []

for f in files:
    df = pd.read_csv(f, sep="\t")
    sample_name = f.split("/")[-1].replace(".tsv","")

    df = df[["probe", "corrected"]]
    df = df.rename(columns={"corrected": sample_name})

    dfs.append(df)

merged = dfs[0]

for df in dfs[1:]:
    merged = merged.merge(df, on="probe")

numeric_cols = merged.columns[1:]

reference = pd.DataFrame({
    "probe": merged["probe"],
    "mean": merged[numeric_cols].mean(axis=1),
    "sd": merged[numeric_cols].std(axis=1)
})

reference.to_csv("reference.tsv", sep="\t", index=False)

print("Reference built")
print(reference.head())

df.to_csv("corrected_counts.tsv", sep="\t", index=False)

Calculate Z-score

In [ ]:
import pandas as pd

sample = pd.read_csv("corrected_counts.tsv", sep="\t")
ref = pd.read_csv("reference.tsv", sep="\t")

df = sample.merge(ref, on="probe")

df["z"] = (df["corrected"] - df["mean"]) / df["sd"]

df.to_csv("zscores.tsv", sep="\t", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("zscores.tsv", sep="\t")

chrom_scores = df.groupby("chrom")["z"].mean()

for chrom, score in chrom_scores.items():
    if score > 3:
        print(f"{chrom}: POSITIVE")
    elif score > 2.5:
        print(f"{chrom}: BORDERLINE")
    else:
        print(f"{chrom}: NEGATIVE")